Date Preparation

In [1]:
import sys
from pathlib import Path

ROOT_DIR = Path().resolve().parent
sys.path.append(str(ROOT_DIR))

from eval_utils import load_docs
import re

docs = load_docs("../data/mk4s_manuel")

cleaned_docs = []

for d in docs:
    text = d.page_content

    # Remove ``` code blocks  
    text = re.sub(r"```.*?```", "", text, flags=re.S)

    # Markdown links [text](url) -> text  
    text = re.sub(r"\[([^\]]+)\]\([^)]+\)", r"\1", text)

    # Remove markdown symbols  
    text = re.sub(r"#+\s*", "", text)
    text = re.sub(r"\*\*", "", text) 

    # Compress multiple blank lines  
    text = re.sub(r"\n{3,}", "\n\n", text)  

    cleaned_docs.append({  
        "source": d.metadata.get("source"),  
        "text": text  
    })  

len(cleaned_docs)  

/home/a.zhang@PTW.Maschinenbau.TU-Darmstadt.de/project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
100%|██████████| 8/8 [00:00<00:00, 9027.29it/s]


8

In [2]:
import spacy

nlp = spacy.load("en_core_web_sm")

sentences = []

for d in cleaned_docs:
    doc = nlp(d["text"])
    for sent in doc.sents:
        sentences.append({
            "source": d["source"],
            "sentence": sent.text.strip()
        })

len(sentences)


617

In [21]:
import csv

with open("Spacy_KG/sentences_full.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["id","text","source",":LABEL"])
    for i, item in enumerate(sentences):
        w.writerow([i, item["sentence"], item["source"], "Sentence"])


pattern-based relation extraction

In [3]:

relations = []

for item in sentences:
    sent = item["sentence"].strip()
    sent_lower = sent.lower()

    if "is caused by" in sent_lower or "are caused by" in sent_lower:
        relations.append({
            "text": sent,
            "type": "cause"
        })

    elif sent_lower.startswith("make sure") or sent_lower.startswith("check"):
        relations.append({
            "text": sent,
            "type": "solution"
        })

    elif "be solved" in sent_lower or " be fixed" in sent_lower:
        relations.append({
            "text": sent,
            "type": "fix"
        })


In [4]:
entity_records = []

for item in relations:
    sent = item["text"]
    sent_type = item["type"]

    doc = nlp(sent)

    for chunk in doc.noun_chunks:
        entity_records.append({
            "entity": chunk.text.strip(),
            "sentence": sent,
            "type": sent_type
        })

len(entity_records)


121

In [5]:
for e in entity_records[:20]:
    print(e)


{'entity': 'the first layer', 'sentence': 'Make sure that the first layer is sticking properly to the entire print surface.', 'type': 'solution'}
{'entity': 'the entire print surface', 'sentence': 'Make sure that the first layer is sticking properly to the entire print surface.', 'type': 'solution'}
{'entity': 'the\xa0Preheat\xa0error', 'sentence': 'Make sure to distinguish between the\xa0Preheat\xa0error and the\xa0Bed preheat\xa0error.', 'type': 'solution'}
{'entity': 'error', 'sentence': 'Make sure to distinguish between the\xa0Preheat\xa0error and the\xa0Bed preheat\xa0error.', 'type': 'solution'}
{'entity': 'the\xa0ambient temperature', 'sentence': 'Make sure that the\xa0ambient temperature is stable\xa0and there is\xa0nothing cooling the printer down, such as a nearby air-conditioning unit or an open window.', 'type': 'solution'}
{'entity': 'nothing', 'sentence': 'Make sure that the\xa0ambient temperature is stable\xa0and there is\xa0nothing cooling the printer down, such as a ne

Filter Entities

In [6]:
import re

def clean_entity(text):
    text = text.lower()
    text = re.sub(r"\b(the|a|an)\b", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

cleaned_entities = []

for e in entity_records:
    cleaned_entities.append({
        "entity": clean_entity(e["entity"]),
        "sentence": e["sentence"],
        "type": e["type"]
    })

cleaned_entities[:10]


[{'entity': 'first layer',
  'sentence': 'Make sure that the first layer is sticking properly to the entire print surface.',
  'type': 'solution'},
 {'entity': 'entire print surface',
  'sentence': 'Make sure that the first layer is sticking properly to the entire print surface.',
  'type': 'solution'},
 {'entity': 'preheat error',
  'sentence': 'Make sure to distinguish between the\xa0Preheat\xa0error and the\xa0Bed preheat\xa0error.',
  'type': 'solution'},
 {'entity': 'error',
  'sentence': 'Make sure to distinguish between the\xa0Preheat\xa0error and the\xa0Bed preheat\xa0error.',
  'type': 'solution'},
 {'entity': 'ambient temperature',
  'sentence': 'Make sure that the\xa0ambient temperature is stable\xa0and there is\xa0nothing cooling the printer down, such as a nearby air-conditioning unit or an open window.',
  'type': 'solution'},
 {'entity': 'nothing',
  'sentence': 'Make sure that the\xa0ambient temperature is stable\xa0and there is\xa0nothing cooling the printer down, such

In [7]:
BAD_ENTITIES = {
    "it", "them", "they", "this", "that", "all", "nothing"
}

def is_valid_entity(text):
    if text in BAD_ENTITIES:
        return False
    if len(text) <= 2:
        return False
    return True

filtered_entities = []

for e in cleaned_entities:
    if is_valid_entity(e["entity"]):
        filtered_entities.append(e)

len(filtered_entities)


112

In [8]:
from collections import defaultdict

sentence_to_entities = defaultdict(list)

for item in filtered_entities:
    sentence_to_entities[item["sentence"]].append(item)

len(sentence_to_entities)


20

In [9]:
for sent, ents in list(sentence_to_entities.items())[:3]:
    print("SENTENCE:", sent)
    print("ENTITIES:", [e["entity"] for e in ents])
    print("-" * 40)


SENTENCE: Make sure that the first layer is sticking properly to the entire print surface.
ENTITIES: ['first layer', 'entire print surface']
----------------------------------------
SENTENCE: Make sure to distinguish between the Preheat error and the Bed preheat error.
ENTITIES: ['preheat error', 'error']
----------------------------------------
SENTENCE: Make sure that the ambient temperature is stable and there is nothing cooling the printer down, such as a nearby air-conditioning unit or an open window.
ENTITIES: ['ambient temperature', 'printer', 'nearby air-conditioning unit', 'open window']
----------------------------------------


In [10]:
import networkx as nx
from itertools import combinations

G = nx.Graph()

for sent, items in sentence_to_entities.items():
    entities = [i["entity"] for i in items]
    sent_type = items[0]["type"]

    # Single-entity sentences also retain nodes
    for ent in entities:
        if not G.has_node(ent):
            G.add_node(ent)

    # Connect every pair of entities within the same sentence
    for e1, e2 in combinations(entities, 2):
        if G.has_edge(e1, e2):

            G[e1][e2]["sentences"].append(sent)
        else:
            G.add_edge(
                e1,
                e2,
                relation="co_occurrence",
                type=sent_type,
                sentences=[sent]
            )

G.number_of_nodes(), G.number_of_edges()


(59, 151)

In [11]:
list(G.nodes)[:20]


['first layer',
 'entire print surface',
 'preheat error',
 'error',
 'ambient temperature',
 'printer',
 'nearby air-conditioning unit',
 'open window',
 'only dish soap',
 'sheet',
 'full side',
 'center',
 'carriage',
 'xy axes movement',
 'belt tension test movements',
 'printer menu',
 'lcd menu -> control',
 'hand',
 'your x/y axis motors',
 'pulleys']

In [12]:
edges = list(G.edges(data=True))
edges[:5]


[('first layer',
  'entire print surface',
  {'relation': 'co_occurrence',
   'type': 'solution',
   'sentences': ['Make sure that the first layer is sticking properly to the entire print surface.']}),
 ('preheat error',
  'error',
  {'relation': 'co_occurrence',
   'type': 'solution',
   'sentences': ['Make sure to distinguish between the\xa0Preheat\xa0error and the\xa0Bed preheat\xa0error.']}),
 ('ambient temperature',
  'printer',
  {'relation': 'co_occurrence',
   'type': 'solution',
   'sentences': ['Make sure that the\xa0ambient temperature is stable\xa0and there is\xa0nothing cooling the printer down, such as a nearby air-conditioning unit or an open window.',
    'Make sure that the\xa0ambient temperature is stable\xa0and there is\xa0cooling down the printer,\xa0such as a nearby air-conditioning unit or an open window.']}),
 ('ambient temperature',
  'nearby air-conditioning unit',
  {'relation': 'co_occurrence',
   'type': 'solution',
   'sentences': ['Make sure that the\xa0am

In [ ]:
import csv
# save documents,sentences,entities as nodes
docs_set = set(d["source"] for d in sentences)

with open("Spacy_KG/documents.csv", "w", encoding="utf-8", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["source", ":LABEL"])
    for s in docs_set:
        writer.writerow([s, "Document"])


In [18]:
with open("Spacy_KG/sentences.csv", "w", encoding="utf-8", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["id", "text", "type", "source", ":LABEL"])

    for i, item in enumerate(relations):
        writer.writerow([
            i,
            item["text"],
            item["type"],
            None,
            "Sentence"
        ])


In [19]:
entities = sorted({e["entity"] for e in filtered_entities})

with open("Spacy_KG/entities.csv", "w", encoding="utf-8", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["name", ":LABEL"])
    for e in entities:
        writer.writerow([e, "Entity"])


In [20]:
# save documents,sentences,entities as nodes
with open("Spacy_KG/doc_sentence.csv", "w", encoding="utf-8", newline="") as f:
    writer = csv.writer(f)
    writer.writerow([":START_ID", ":END_ID", ":TYPE"])

    for i, s in enumerate(sentences):
        writer.writerow([
            s["source"],
            i,
            "CONTAINS"
        ])

with open("Spacy_KG/sentence_entity.csv", "w", encoding="utf-8", newline="") as f:
    writer = csv.writer(f)
    writer.writerow([":START_ID", ":END_ID", ":TYPE"])

    for item in filtered_entities:
        writer.writerow([
            item["sentence"],
            item["entity"],
            "MENTIONS"
        ])


Open in Neo4j
// 1. Documents
LOAD CSV WITH HEADERS FROM 'file:///documents.csv' AS row
MERGE (:Document {source: row.source});

// 2. Sentences
LOAD CSV WITH HEADERS FROM 'file:///sentences.csv' AS row
MERGE (:Sentence {id: toInteger(row.id)})
SET s.text = row.text,
    s.type = row.type;

// 3. Entities
LOAD CSV WITH HEADERS FROM 'file:///entities.csv' AS row
MERGE (:Entity {name: row.name});

// 4. Document → Sentence
LOAD CSV WITH HEADERS FROM 'file:///doc_sentence.csv' AS row
MATCH (d:Document {source: row.:START_ID})
MATCH (s:Sentence {id: toInteger(row.:END_ID)})
MERGE (d)-[:CONTAINS]->(s);

// 5. Sentence → Entity
LOAD CSV WITH HEADERS FROM 'file:///sentence_entity.csv' AS row
MATCH (s:Sentence {text: row.:START_ID})
MATCH (e:Entity {name: row.:END_ID})
MERGE (s)-[:MENTIONS]->(e);
